# Intro

- Recoding of https://github.com/Daniel-EST/deep-steganography/ but updated to match modern tensorflow
- using dahuffman for huffman and [LSBSteg repo](https://github.com/RobinDavid/LSB-Steganography/tree/master) for training with huffman+lsb secret inside initial secret image


In [ ]:
import tensorflow as tf
from PIL import Image
import numpy as np
from LSB_Steganography.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# from https://github.com/RobinDavid/LSB-Steganography/tree/master


def LSBEmbed(huffman_text, image):
    img_array = np.array(image.convert("RGB"))
    steg = LSBSteg(img_array)
    return Image.fromarray(steg.encode_binary(huffman_text))


def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    if isinstance(image, np.ndarray):
        img_array = image
    else:
        img_array = np.array(image.convert("RGB"))

    steg = LSBSteg(img_array)
    return steg.decode_binary()

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example


def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data


def HuffmanDecode(codec, data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
tf.config.optimizer.set_jit(True)

In [ ]:
from datasets import load_dataset
import certifi

os.environ['SSL_CERT_FILE'] = certifi.where()


def is_rgb(example):
    return example['image'].mode == 'RGB'


# option 1 uses "Hello World!" for all to reuse codec and make things simpler
temp_data = load_dataset("zh-plus/tiny-imagenet")
tiny_imageNet_rgb = temp_data.filter(is_rgb)

# based on original repo 7500 secret + cover so 15k total
num_samples = 15000  # len(tiny_imageNet_rgb['train'])
train_pool = tiny_imageNet_rgb['train'].select(range(num_samples))
test_pool = tiny_imageNet_rgb['valid']
train_pool = train_pool.shuffle(seed=42)
test_pool = test_pool.shuffle(seed=42)

ex_data = "Hello World!"
codec, encoded_data = HuffmanEncode(ex_data)


def make_dataset_pools(dataset, huffman_data):
    secret_pool = []
    cover_pool = []

    # Iterate through the dataset in pairs
    for i in range(0, len(dataset) - 1, 2):
        # Resize to 64x64 as per your model architecture
        img_a = dataset[i]['image'].resize((64, 64))
        img_b = dataset[i+1]['image'].resize((64, 64))

        # Create the 'secret_input' (Image A + Hidden Data)
        # dont embed now just to reproduce original NN
        secret_input_val = np.array(img_b).astype(
            'float32') / 255.0  # LSBEmbed(huffman_data, img_a)
        # secret_input_val = np.array(stego_pil_image).astype('float32') / 255.0

        # Create the 'cover_input' (Clean Image B)
        cover_input_val = np.array(img_b).astype('float32') / 255.0

        secret_pool.append(secret_input_val)
        cover_pool.append(cover_input_val)

    return np.array(secret_pool), np.array(cover_pool)


# Generate the final arrays
train_secret, train_cover = make_dataset_pools(train_pool, encoded_data)
test_secret, test_cover = make_dataset_pools(test_pool, encoded_data)

# sample_to_check = (train_secret[0] * 255).astype('uint8')
# extracted_bits = LSBExtract(sample_to_check)

# if encoded_data == extracted_bits:
#     print("✅ Data integrity confirmed! Ready for training.")
# else:
#     print("❌ Warning: Data mismatch. Check encoding format.")

### Layers copied from keras tensor impl of original


In [ ]:
model.summary()
# 5. Plot with expand_nested=False
tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_dtype=False,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96,
    layer_range=None,
    show_layer_activations=False
)

In [ ]:
# --- Training Configuration ---
import time
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001

model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE), jit_compile=True)

# slight change from original but should be better
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=5,      # Wait 5 epochs
    min_lr=1e-6,     # Don't go below this
    verbose=1
)
callbacks = [
    # lr_scheduler,
    tf.keras.callbacks.EarlyStopping(parience=10),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='./checkpoints/model.{epoch:02d}-{val_loss:.2f}.h5'),
    tf.keras.callbacks.TensorBoard(log_dir='./logs')
]

# samples / 2 (because 1 conver + 1 secret / batch)
start_time = time.time()
history = model.fit(
    ([train_secret, train_cover], test_secret),
    validation_data=([test_secret, test_cover], test_secret),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[callbacks],
    verbose=1, shuffle=True
)
# think this is needed to save weights?
end_time = time.time()

total_time = end_time - start_time
print("Training Complete.")
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)
seconds = int(total_time % 60)

epoch_hours = int(total_time/EPOCHS // 3600)
epoch_minutes = int((total_time/EPOCHS % 3600) // 60)
epoch_seconds = int(total_time/EPOCHS % 60)

print(
    f"Total training time: {hours}h {minutes}m {seconds}s, time/epoch: {epoch_hours}h {epoch_minutes}m {epoch_seconds}s")

In [ ]:
# Save the whole model weights
model.save('steganography_model.keras')

In [ ]:
import matplotlib.pyplot as plt
predictions = mode.predict([])